In [2]:
from langchain_openai import ChatOpenAI

In [1]:
api_key="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"

In [3]:
llm= ChatOpenAI(api_key="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",
                base_url="https://openrouter.ai/api/v1",
                model_name= "openrouter/free",
                temperature=0.7)

In [4]:
from langchain_classic.prompts import PromptTemplate

In [22]:
from langchain_classic.chains import ConversationChain
from langchain_classic.memory import ConversationBufferMemory

In [24]:
chat= ConversationChain(llm=llm, memory = ConversationBufferMemory())

In [26]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import TokenTextSplitter

## Loading data

In [27]:
loader = PyPDFLoader(r"D:\NLP intern\TASKS\TASK3\Mohamed_Elwan_CV.pdf")
documents=loader.load()

## Split

In [29]:
splitter = TokenTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

## embedding

In [30]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",model_kwargs = {"device":"cpu"})
vectors = embeddings.embed_documents([i.page_content for i in chunks])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## vectorDB

In [32]:
vectorDB= FAISS.from_documents(chunks, embeddings)

## RAG

In [34]:
temp = """
you are assistant, Answer the next Question using provided context,
If you don't know the answer, just say you don't know.
answer should be within 150 words or lower only
## context :
{context}
## Question :
{Question}
"""
temp = PromptTemplate.from_template(temp)

query = input("Enter your question: ")
similar_docs = vectorDB.similarity_search_with_score(query,k=2)
context = []

for i in similar_docs :
    context.append(i[0].page_content)
    prompt = temp.format(context="\n".join(context), Question = query)

response = chat.predict(input=prompt)
print(response)

Your education is:

- **Degree:** B.Sc. in Computer & Systems Engineering  
- **Institution:** Ain Shams University – Faculty of Engineering  
- **Location:** Cairo, Egypt  
- **Duration:** September 2022 – July 2027 (currently in progress)
